# Serving Qwen3 4B với Ollama trên Colab + Inference bằng OpenAI Python

Notebook này thực hiện:
1. Cài đặt **Ollama** trên Colab
2. Khởi chạy Ollama server ở background
3. Pull model **`qwen3.5:4b`** (model 4B của họ Qwen3.5)
4. Gọi inference qua **OpenAI Python client** (Ollama có endpoint tương thích OpenAI tại `/v1`)

> Gợi ý: Vào `Runtime > Change runtime type > T4 GPU` để chạy nhanh hơn. Ollama vẫn chạy được trên CPU nhưng chậm hơn.

## 1. Cài đặt Ollama

In [1]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,900 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current 

## 2. Khởi chạy Ollama server ở background

Chạy `ollama serve` trong một tiến trình nền và đợi cho tới khi API sẵn sàng tại `http://localhost:11434`.

In [1]:
import subprocess, os, time, requests


# Khởi chạy server ở background, log ra file để debug nếu cần
log_file = open("ollama.log", "w")
server_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
)

# Đợi server sẵn sàng
def wait_for_server(url="http://localhost:11434", timeout=60):
    start = time.time()
    while time.time() - start < timeout:
        try:
            if requests.get(url).status_code == 200:
                print("Ollama server đã sẵn sàng!")
                return True
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(1)
    raise RuntimeError("Ollama server không khởi động được trong thời gian cho phép.")

wait_for_server()

Ollama server đã sẵn sàng!


True

## 3. Pull model `qwen3.5:4b`

Lần đầu sẽ mất vài phút để tải model. Bạn có thể đổi `MODEL_NAME` sang tag khác, ví dụ `qwen3:4b`, `qwen2.5:3b`...

In [4]:
MODEL_NAME = "qwen3.5:4b"

In [3]:
MODEL_NAME = "qwen3.5:4b"
!ollama pull {MODEL_NAME}

In [2]:
# Kiểm tra model đã có
!ollama list

NAME          ID              SIZE      MODIFIED       
qwen3.5:4b    2a654d98e6fb    3.4 GB    26 minutes ago    


## 4. Cài OpenAI Python client

In [ ]:
!pip show openai

Name: openai
Version: 2.41.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: 


In [ ]:
!pip install -q openai

## 5. Inference đơn giản qua OpenAI client

Ollama expose một endpoint tương thích OpenAI tại `http://localhost:11434/v1`. `api_key` chỉ là placeholder (Ollama không kiểm tra) nhưng bắt buộc phải có giá trị.

In [5]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # bắt buộc nhưng không được kiểm tra
)

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "Bạn là một trợ lý hữu ích, trả lời ngắn gọn bằng tiếng Việt."},
        {"role": "user", "content": "Giải thích ngắn gọn machine learning là gì?"},
    ],
    temperature=0.7,
)

print(response.choices[0].message.content)

Machine Learning là một nhánh của Trí tuệ nhân tạo (AI), giúp máy tính tự động học tập và đưa ra dự đoán thông qua việc phân tích dữ liệu, thay vì được lập trình bằng các quy tắc cứng nhắc truyền thống.


## 6. (Tuỳ chọn) Streaming output

In token ngay khi model sinh ra, thay vì đợi toàn bộ câu trả lời.

In [6]:
stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Viết một đoạn code Python tính giai thừa bằng đệ quy."},
    ],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

Dưới đây là đoạn code Python để tính giai thừa ($n!$) bằng phương pháp đệ quy (Recursion).

### Giải thích logic ngắn gọn:
Giai thừa $n! = n \times (n-1)!$. Điều này có nghĩa để giải cho số lớn, program sẽ gọi chính nó với một giá trị nhỏ hơn xuống đến khi đạt điều kiện dừng cơ bản ($0!$ hoặc $1! = 1$).

```python
def tinh_giai_thua(n):
    """Tính giai thừa của n bằng cách đệ quy. 
    
    - Case base: Nếu n <= 1 thì trả về 1 (vì 0!=1, 1!=1)*/ 
        if n < 2: # Khi n == 0 hoặc n == 1
            return 1
    
    else:           # Bước đệ quy: n! = n * (n-1)!
        return n * tinh_giai_thua(n - 1)


# ==========================================
# Ví dụ sử dụng hàm (Call function here below):
print("Giai thừa của số:", int(input("Nhập một số nguyên dương để tính giai thừa: ", type=int))) # Input user cho đẹp

s = 5 # Kết quả cần dùng test cụ thể là ví dụ này nhé nếu không có input từ người dùng trong vòng lặp code như trên.
result = tinh_giai_thua(s) 
print(f"Giai thừa của {s} (!{

## 7. Dự đoán góc quay


In [6]:
!pip install python-magic==0.4.27

In [7]:
import base64
import magic
def encode_image(image_path_or_url: str, get_mime_type: bool = False):
    """Encode image (URL hoặc local path) -> base64.
    Trả về tuple (b64, mime) nếu get_mime_type=True, ngược lại chỉ trả b64."""

    def _fail():
        return (None, None) if get_mime_type else None

    if image_path_or_url.startswith("http"):
        try:
            response = requests.get(
                image_path_or_url,
                stream=True,
                timeout=15
            )
            response.raise_for_status()

            mime_type = response.headers.get("content-type", None)
            b64 = base64.b64encode(response.content).decode("utf-8")

            return (b64, mime_type) if get_mime_type else b64

        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
            return _fail()

    else:
        if not os.path.exists(image_path_or_url):
            return _fail()

        mime_type = magic.Magic(mime=True).from_file(image_path_or_url)

        if not mime_type.startswith("image/"):
            return _fail()

        with open(image_path_or_url, "rb") as image_file:
            b64 = base64.b64encode(
                image_file.read()
            ).decode("utf-8")

        return (b64, mime_type) if get_mime_type else b64

In [8]:
# === SYSTEM PROMPT ===
SYSTEM_PROMPT = """
Bạn là bộ điều khiển thị giác (vision-based controller) cho cánh tay robot 6 bậc tự do SO-101 trong môi trường mô phỏng MuJoCo. Nhiệm vụ của bạn là QUAN SÁT hai ảnh camera mô tả hiện trạng, rồi DỰ ĐOÁN góc quay mục tiêu (radian) cho 6 khớp để cánh tay tiếp cận và GẮP đúng vật thể được yêu cầu.

Nguyên tắc bắt buộc:
1. Chỉ suy luận dựa trên những gì NHÌN THẤY trong 2 ảnh, không bịa thông tin không quan sát được.
2. Mọi góc đầu ra phải là radian và NẰM TRONG giới hạn cho phép của từng khớp. Tuyệt đối không vượt biên.
3. Điều khiển theo VÒNG KÍN: mỗi lần chỉ đề xuất bước di chuyển an toàn, nhỏ vừa phải tiến gần mục tiêu; ở bước cuối mới đóng kẹp.
4. Chỉ đóng kẹp khi đầu kẹp đã thẳng hàng và đủ gần vật trong ảnh.
5. Đầu ra DUY NHẤT là một đối tượng JSON hợp lệ, không kèm văn bản nào khác, không markdown, không khối.
"""

# === USER PROMPT ===
USER_PROMPT = """
# MỤC TIÊU
Điều khiển cánh tay SO-101 để gắp: **khối lập phương màu đỏ (cube_red)**.

# BỐI CẢNH 2 CAMERA
Bạn nhận được 2 ảnh theo thứ tự:
- Ảnh 1 — CAMERA CỔ TAY (wrist_cam): gắn cứng trên kẹp, nhìn cận cảnh vùng kẹp gắp. Dùng để căn chính xác đầu kẹp so với vật (trái/phải, trước/sau, đã thẳng hàng chưa, đã đủ gần để đóng kẹp chưa).
- Ảnh 2 — CAMERA GIÁ ĐỠ (stand_cam): đặt cố định bên cạnh, nhìn ngang toàn cảnh. Dùng để ước lượng tư thế tổng thể của cánh tay, độ cao đầu kẹp và khoảng cách thô tới vật.

# CƠ CẤU CHẤP HÀNH (6 khớp position-controlled, đơn vị RADIAN)
Mỗi khớp là một động cơ vị trí: bạn đặt GÓC MỤC TIÊU tuyệt đối, bộ điều khiển PID nội bộ sẽ đưa khớp tới góc đó. Tư thế "home" là tư thế nghỉ an toàn để tham chiếu.

| # | Tên khớp | Giới hạn (radian) | Home | Mô tả |
|---|----------|-------------------|------|-------|
| 1 | `shoulder_pan` | [-1.920, 1.920] | 0.00 | Xoay toàn thân cánh tay quanh trục thẳng đứng (z). Giá trị + xoay thân sang một bên, - xoay sang bên kia. Dùng để nhắm cánh tay theo phương ngang về phía vật. |
| 2 | `shoulder_lift` | [-1.745, 1.745] | -0.80 | Nâng/hạ vai (đoạn cánh tay trên). Giá trị + nâng vai lên, - hạ vai xuống gần mặt sàn. |
| 3 | `elbow_flex` | [-1.690, 1.690] | 1.20 | Gập/duỗi khuỷu tay. Điều chỉnh tầm với (xa/gần) và độ cao của đầu kẹp. |
| 4 | `wrist_flex` | [-1.658, 1.658] | 0.90 | Gập cổ tay lên/xuống. Dùng để chỉnh phương tiếp cận của kẹp (gắp từ trên xuống hay từ ngang vào). |
| 5 | `wrist_roll` | [-2.744, 2.841] | 1.57 | Xoay cổ tay quanh trục kẹp. Dùng để căn hướng của hai má kẹp cho khớp với chiều dài/hướng của vật. |
| 6 | `gripper` | [-0.175, 1.745] | 1.20 | Độ mở của kẹp. Giá trị LỚN = mở rộng, giá trị NHỎ (gần giới hạn dưới) = đóng/kẹp chặt. |

Quy ước về kẹp (`gripper`): giá trị càng LỚN càng MỞ rộng, càng NHỎ (về phía giới hạn dưới) càng ĐÓNG chặt.

# QUY TRÌNH GẮP (các pha)
- "approach": di chuyển đầu kẹp lại gần vật, kẹp đang MỞ.
- "align": tinh chỉnh shoulder_pan / wrist_roll / wrist_flex để đầu kẹp thẳng hàng và bao quanh vật.
- "grasp": đầu kẹp đã ôm vật -> ĐÓNG kẹp.
- "lift": nâng shoulder_lift / elbow_flex để nhấc vật lên khỏi sàn.
Chọn đúng pha hiện tại dựa trên những gì thấy trong ảnh.

# ĐỊNH DẠNG ĐẦU RA (chỉ JSON, không thêm gì khác)
Trả về đúng cấu trúc sau:
- "reasoning": (string) suy luận ngắn gọn từ 2 ảnh.
- "target_object": (string) tên/định danh vật đang nhắm tới.
- "phase": một trong ["approach","align","grasp","lift"].
- "gripper_action": "open" | "close" | "hold".
- "joint_angles_rad": (object) góc mục tiêu radian cho ĐỦ 6 khớp: "shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll", "gripper".
- "done": (bool) true khi đã gắp và nâng vật thành công.
- "confidence": (float 0..1) mức độ chắc chắn.

Ví dụ một đầu ra hợp lệ:
{
  "reasoning": "Mô tả ngắn gọn vị trí vật trong 2 ảnh và cách cánh tay cần di chuyển để tiếp cận và kẹp.",
  "target_object": "cube_red",
  "phase": "approach",
  "gripper_action": "open",
  "joint_angles_rad": {
    "shoulder_pan": 0.0,
    "shoulder_lift": -0.8,
    "elbow_flex": 1.2,
    "wrist_flex": 0.9,
    "wrist_roll": 1.57,
    "gripper": 1.2
  },
  "done": false,
  "confidence": 0.0
}

Bây giờ hãy phân tích 2 ảnh và xuất JSON điều khiển cho bước tiếp theo.
"""

wrist_img_b64, wrist_mime_type = encode_image("/content/camera_1_co_tay_wrist_20260615_144722.png", get_mime_type=True)
stand_img_b64, stand_mime_type = encode_image("/content/camera_2_gia_do_stand_20260615_144724.png", get_mime_type=True)

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT,
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": USER_PROMPT,
            },
            {
                "type": "text",
                "text": "Ảnh 1 — CAMERA CỔ TAY (wrist_cam):",
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{wrist_mime_type};base64,{wrist_img_b64}",
                },
            },
            {
                "type": "text",
                "text": "Ảnh 2 — CAMERA GIÁ ĐỠ (stand_cam):",
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{stand_mime_type};base64,{stand_img_b64}",
                },
            },
        ],
    },
]

### 7.1 Sử dụng OpenAI client

In [26]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # bắt buộc nhưng không được kiểm tra
)


In [13]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    max_tokens=120000,
    temperature=0.8,
    top_p=0.1,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
    },
)

print(response.choices[0].message.content)

In [14]:
print(response.choices[0].finish_reason)
print(repr(response.choices[0].message.content))
# một số bản Ollama trả reasoning ở đây:
print(getattr(response.choices[0].message, "reasoning_content", None))
print(getattr(response.choices[0].message, "reasoning", None))

length
''
None
The user wants me to act as a vision-based controller for a SO-101 robot arm in MuJoCo.

**Input Analysis:**
*   **Image 1 (Wrist Cam):** Shows the gripper very close to an object. The object is a red cube (`cube_red`). It looks like it's slightly behind or below the tip of the fingers, but definitely within reach. The gripper jaws are open and positioned right next to the corner/edge of the red block.
*   **Image 2 (Stand Cam):** Shows the full robot arm reaching towards a group of objects on the floor. There is a green cylinder, a blue cube, and a red cube (`cube_red`). The red cube is clearly visible in front/left of the others relative to the camera view. The gripper is currently hovering above/near these objects but hasn't grasped anything yet (or just finished an approach).

**Current State Assessment:**
*   **Target Object:** `cube_red`. It's sitting on the floor.
*   **Phase:** Looking at Image 1, the gripper is extremely close to the red cube. The fingers are op

In [16]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    max_tokens=2048,
    temperature=0.2,
    top_p=0.9,
    presence_penalty=0.0,
    response_format={"type": "json_object"},   # ép xuất JSON hợp lệ
    extra_body={
        "think": False,                        # TẮT chế độ suy luận của Qwen3/3.5
        "options": {"num_ctx": 8192},
    },
)

msg = response.choices[0].message
print("finish:", response.choices[0].finish_reason)
print(msg.content)


finish: length



In [17]:
r = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Trả lời đúng 1 từ: thủ đô Việt Nam?"}],
    max_tokens=512,
    extra_body={"think": False, "options": {"num_ctx": 4096}},
)
print(r.choices[0].finish_reason, "|", repr(r.choices[0].message.content))


length | ''


### 7.2 Sử dụng Ollama client

In [19]:
!pip install ollama

In [20]:
import ollama

resp = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "/no_think Trả lời đúng 1 từ: thủ đô Việt Nam?"}
    ],
    think=False,
    options={"num_ctx": 4096, "num_predict": 256, "temperature": 0.2},
)
print(repr(resp["message"]["content"]))
print("thinking:", resp["message"].get("thinking"))


'Hà Nội'
thinking: None


In [22]:
import ollama, json

resp = ollama.chat(
    model=MODEL_NAME,                    # qwen3.5:4b vẫn dùng được vì nó nhìn được ảnh
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",
         "content": USER_PROMPT,
         "images": [
             "/content/camera_1_co_tay_wrist_20260615_144722.png",
             "/content/camera_2_gia_do_stand_20260615_144724.png",
         ]},
    ],
    think=False,                         # quan trọng: tắt reasoning để content có dữ liệu
    format="json",                       # ép JSON hợp lệ
    options={"num_ctx": 16384, "num_predict": 2048, "temperature": 0.2},
)

content = resp["message"]["content"]
# Remove markdown code block wrapper if present
if content.startswith('```json') and content.endswith('```'):
    content = content[len('```json'):-len('```')].strip()

print(content)

# Parse ra dict để dùng trong vòng điều khiển
action = json.loads(content)
print(action["phase"], action["gripper_action"])
print(action["joint_angles_rad"])


{
  "reasoning": "Camera 1 (Cổ tay) hiện tại đang hướng xuống dưới gần sàn, kẹp mở rộng ra xa vật thể màu đỏ nằm trên mặt phẳng ngang. Camera 2 (Giá đỡ) xác nhận vị trí của khối lập phương màu đỏ so với cánh tay và cho thấy đầu kẹp chưa ở tư thế đúng để tiếp cận. Cánh tay cần nâng lên cao hơn hiện tại và xoay thân sang phải để hướng về phía vật.",
  "target_object": "cube_red",
  "phase": "approach",
  "gripper_action": "open",
  "joint_angles_rad": {
    "shoulder_pan": 1.0,
    "shoulder_lift": -0.5,
    "elbow_flex": 1.2,
    "wrist_flex": 0.9,
    "wrist_roll": 1.57,
    "gripper": 1.2
  },
  "done": false,
  "confidence": 0.8
}
approach open
{'shoulder_pan': 1.0, 'shoulder_lift': -0.5, 'elbow_flex': 1.2, 'wrist_flex': 0.9, 'wrist_roll': 1.57, 'gripper': 1.2}


In [24]:
import json, re

def parse_json(content: str) -> dict:
    # bỏ rào ```json ... ``` nếu có
    content = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", content.strip())
    # phòng hờ: cắt đúng object JSON ngoài cùng
    start, end = content.find("{"), content.rfind("}")
    if start != -1 and end != -1:
        content = content[start:end + 1]
    return json.loads(content)

action = parse_json(resp["message"]["content"])
print(action["phase"], action["gripper_action"])
print(action["joint_angles_rad"])


approach open
{'shoulder_pan': 1.0, 'shoulder_lift': -0.5, 'elbow_flex': 1.2, 'wrist_flex': 0.9, 'wrist_roll': 1.57, 'gripper': 1.2}


In [23]:
import math

joint_angles_rad = {'shoulder_pan': 1.0, 'shoulder_lift': -0.5, 'elbow_flex': 1.2, 'wrist_flex': 0.9, 'wrist_roll': 1.57, 'gripper': 1.2}

joint_angles_deg = {}
for joint, rad_value in joint_angles_rad.items():
    joint_angles_deg[joint] = math.degrees(rad_value)

print(joint_angles_deg)

{'shoulder_pan': 57.29577951308232, 'shoulder_lift': -28.64788975654116, 'elbow_flex': 68.75493541569878, 'wrist_flex': 51.56620156177409, 'wrist_roll': 89.95437383553924, 'gripper': 68.75493541569878}
